# Module 11: Applications -- KV Cache & Vector Search

*Estimated time: ~35 minutes*

You have built the full TurboQuant system from scratch and proved it is near-optimal.
Now the question is: **where does this actually matter?**

This module explores two high-impact applications where TurboQuant's unique properties --
zero training time, unbiased inner products, and near-optimal compression -- translate
directly into practical wins:

1. **KV Cache Compression** -- shrinking the memory bottleneck in transformer inference
2. **Vector Search** -- building an approximate nearest neighbor index with zero indexing time

Both applications exploit the same core insight: TurboQuant preserves inner products.
Attention is inner products. Similarity search is inner products. It all connects.

In [ ]:
import sys; sys.path.insert(0, "..")
from src.helpers import section_header, check_intuition

import time
import numpy as np
import torch
import torch.nn.functional as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots

torch.manual_seed(42)

---
## Part 1: KV Cache Compression

### 1. What is a KV cache and why does it matter?

When a transformer generates text token by token, it performs **attention** at every layer.
Each attention head computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d}}\right) V$$

where:
- **Q** (Query) comes from the current token being generated
- **K** (Keys) come from all previous tokens
- **V** (Values) come from all previous tokens

The critical insight: when generating token 1001, you need Keys and Values from tokens
1 through 1000. Recomputing them every step would be wasteful, so transformers **cache**
them. This is the **KV cache**.

The shape of each cached tensor is:

```
(batch, num_heads, sequence_length, head_dim)
```

Typically head_dim = 128. The problem? **Memory grows linearly with sequence length.**

In [ ]:
section_header(
    "KV Cache Memory Problem",
    "Memory grows linearly with sequence length"
)

# A realistic model configuration (roughly LLaMA-70B scale)
batch = 1
num_heads = 64
head_dim = 128
bytes_per_float = 4  # float32

print(f"Model config: {num_heads} heads, head_dim={head_dim}, float32\n")
print(f"{'Seq Length':<15} {'KV Cache (K+V)':<20} {'Notes'}")
print("-" * 60)

for seq_len in [1_000, 10_000, 50_000, 100_000, 200_000]:
    # K and V each: batch * heads * seq * head_dim * 4 bytes
    kv_bytes = batch * num_heads * seq_len * head_dim * bytes_per_float * 2  # x2 for K+V
    kv_gb = kv_bytes / (1024 ** 3)
    notes = ""
    if kv_gb > 10:
        notes = "<-- exceeds many GPU memory budgets"
    elif kv_gb > 1:
        notes = "<-- significant fraction of GPU memory"
    print(f"  {seq_len:>10,}   {kv_gb:>10.2f} GB       {notes}")

print("\nLong contexts are increasingly common (128K+ for modern models).")
print("The KV cache often dominates GPU memory during inference.")

### 2. The attention formula is all inner products

Look at the attention formula again:

$$\text{softmax}\left(\frac{Q K^T}{\sqrt{d}}\right) V$$

The $Q K^T$ part computes **inner products** between the query and every key.
These inner products become the attention logits (before softmax).

This is exactly what TurboQuant preserves! Specifically:

- **Keys** need unbiased inner products (for accurate attention logits)
  -> use `unbiased=True`
- **Values** need low reconstruction error (for output quality)
  -> use `unbiased=False`

No training needed. Works with any model, any sequence length. Compress at 3-4 bits
for 8-10x memory savings.

Let's build this from scratch.

---
### 3. Build a toy KV cache compressor

We will simulate a transformer's KV cache, compress it with TurboQuant,
and compare the attention output to the uncompressed version.

Since we are building this from scratch (not importing the reference code),
we will implement a simplified version using the core TurboQuant class.

In [ ]:
section_header(
    "Building a KV Cache Compressor",
    "Simulate, compress, and compare attention outputs"
)

# Add reference code to path so we can use the TurboQuant implementation
sys.path.insert(0, "../reference-code/turboquant-torch")
from turboquant.core import TurboQuant

# Simulate a transformer's KV cache
batch, heads, seq_len, head_dim = 1, 8, 512, 128

torch.manual_seed(42)
keys = torch.randn(batch, heads, seq_len, head_dim)
values = torch.randn(batch, heads, seq_len, head_dim)
query = torch.randn(batch, heads, 1, head_dim)  # single new token

print(f"KV cache shape: ({batch}, {heads}, {seq_len}, {head_dim})")
print(f"Query shape:    ({batch}, {heads}, 1, {head_dim})")
print(f"\nOriginal KV memory: {keys.numel() * 4 + values.numel() * 4:,} bytes")
print(f"  = {(keys.numel() + values.numel()) * 4 / 1024:.1f} KB")

In [ ]:
# Step 1: Compute ground truth attention (uncompressed)
scale = head_dim ** -0.5

# Q @ K^T -> attention logits
attn_logits_orig = torch.matmul(query, keys.transpose(-2, -1)) * scale  # (B, H, 1, S)
attn_weights_orig = F.softmax(attn_logits_orig, dim=-1)                 # (B, H, 1, S)
attn_output_orig = torch.matmul(attn_weights_orig, values)              # (B, H, 1, D)

print("Ground truth attention computed.")
print(f"Attention logits shape: {attn_logits_orig.shape}")
print(f"Attention weights shape: {attn_weights_orig.shape}")
print(f"Attention output shape:  {attn_output_orig.shape}")

In [ ]:
# Step 2: Compress KV cache with TurboQuant
#
# Keys: unbiased=True  (accurate inner products for attention logits)
# Values: unbiased=False (lower MSE for output reconstruction)

bit_width = 3
key_quantizer = TurboQuant(head_dim, bit_width=bit_width, unbiased=True, seed=0)
val_quantizer = TurboQuant(head_dim, bit_width=bit_width, unbiased=False, seed=100)

# Flatten to (B*H*S, D), quantize, then reshape back
B, H, S, D = keys.shape
k_flat = keys.reshape(-1, D)
v_flat = values.reshape(-1, D)

k_compressed = key_quantizer.quantize(k_flat)
v_compressed = val_quantizer.quantize(v_flat)

# Decompress back to original shape
k_decompressed = key_quantizer.dequantize(k_compressed).view(B, H, S, D)
v_decompressed = val_quantizer.dequantize(v_compressed).view(B, H, S, D)

print(f"Compressed at {bit_width} bits per coordinate.")
print(f"Keys reconstruction MSE:   {(keys - k_decompressed).pow(2).mean():.6f}")
print(f"Values reconstruction MSE:  {(values - v_decompressed).pow(2).mean():.6f}")

---
### 4. Compare attention outputs

Now the real test: does compressed attention give the same result as uncompressed?

In [ ]:
section_header(
    "Attention Quality Comparison",
    "Compressed vs. uncompressed attention"
)

# Compute attention with compressed KV cache
attn_logits_comp = torch.matmul(query, k_decompressed.transpose(-2, -1)) * scale
attn_weights_comp = F.softmax(attn_logits_comp, dim=-1)
attn_output_comp = torch.matmul(attn_weights_comp, v_decompressed)

# Metric 1: Attention weight cosine similarity
# Flatten weights to (B*H, S) for cosine similarity
w_orig_flat = attn_weights_orig.reshape(-1, S)
w_comp_flat = attn_weights_comp.reshape(-1, S)
cos_sim = F.cosine_similarity(w_orig_flat, w_comp_flat, dim=-1)

# Metric 2: Output MSE
output_mse = (attn_output_orig - attn_output_comp).pow(2).mean().item()

# Metric 3: Output cosine similarity
out_orig_flat = attn_output_orig.reshape(-1, D)
out_comp_flat = attn_output_comp.reshape(-1, D)
output_cos_sim = F.cosine_similarity(out_orig_flat, out_comp_flat, dim=-1)

print(f"At {bit_width} bits per coordinate:\n")
print(f"Attention weight cosine similarity:  {cos_sim.mean():.6f}  (1.0 = perfect)")
print(f"Output MSE:                          {output_mse:.6f}")
print(f"Output cosine similarity:            {output_cos_sim.mean():.6f}  (1.0 = perfect)")
print(f"\nAt 3 bits, the compressed attention is very close to the original.")

In [ ]:
# Sweep across bit widths to see the quality vs compression trade-off
results_kv = {}

for bw in [2, 3, 4]:
    kq = TurboQuant(head_dim, bit_width=bw, unbiased=True, seed=0)
    vq = TurboQuant(head_dim, bit_width=bw, unbiased=False, seed=100)

    kc = kq.quantize(k_flat)
    vc = vq.quantize(v_flat)

    k_dec = kq.dequantize(kc).view(B, H, S, D)
    v_dec = vq.dequantize(vc).view(B, H, S, D)

    logits = torch.matmul(query, k_dec.transpose(-2, -1)) * scale
    weights = F.softmax(logits, dim=-1)
    output = torch.matmul(weights, v_dec)

    w_cos = F.cosine_similarity(
        attn_weights_orig.reshape(-1, S), weights.reshape(-1, S), dim=-1
    ).mean().item()

    o_mse = (attn_output_orig - output).pow(2).mean().item()
    o_cos = F.cosine_similarity(
        attn_output_orig.reshape(-1, D), output.reshape(-1, D), dim=-1
    ).mean().item()

    results_kv[bw] = {"weight_cos": w_cos, "output_mse": o_mse, "output_cos": o_cos}

print(f"{'Bits':<8} {'Weight CosSim':<18} {'Output MSE':<16} {'Output CosSim':<16}")
print("-" * 58)
for bw, r in results_kv.items():
    print(f"  {bw:<6} {r['weight_cos']:<18.6f} {r['output_mse']:<16.6f} {r['output_cos']:<16.6f}")

In [ ]:
# Visualize: attention weight distributions for one head (original vs compressed)
head_idx = 0
w_orig_h0 = attn_weights_orig[0, head_idx, 0, :].detach().numpy()
w_comp_h0 = attn_weights_comp[0, head_idx, 0, :].detach().numpy()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Attention Weights (Head 0)", "Weight Difference"),
    horizontal_spacing=0.12,
)

# Show a window of the sequence for readability
window = slice(0, 100)
x_pos = np.arange(window.start, window.stop)

fig.add_trace(
    go.Scatter(x=x_pos, y=w_orig_h0[window], mode="lines", name="Original",
               line=dict(color="#3498db", width=1.5)),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=x_pos, y=w_comp_h0[window], mode="lines", name="Compressed (3-bit)",
               line=dict(color="#e74c3c", width=1.5, dash="dash")),
    row=1, col=1,
)

diff = w_orig_h0[window] - w_comp_h0[window]
fig.add_trace(
    go.Bar(x=x_pos, y=diff, name="Difference",
           marker_color=["#e74c3c" if d > 0 else "#3498db" for d in diff]),
    row=1, col=2,
)

fig.update_layout(
    title="Attention Weight Comparison: Original vs. 3-bit Compressed",
    template="plotly_white", height=400, width=900,
    showlegend=True,
)
fig.update_xaxes(title_text="Token Position", row=1, col=1)
fig.update_xaxes(title_text="Token Position", row=1, col=2)
fig.update_yaxes(title_text="Weight", row=1, col=1)
fig.update_yaxes(title_text="Weight Diff", row=1, col=2)
fig.show()

---
### 5. Memory savings calculation

Let's do the math on how much memory we actually save.

In [ ]:
section_header(
    "Memory Savings",
    "How much memory does KV cache compression save?"
)

def kv_memory_original(batch, heads, seq_len, head_dim, bytes_per_elem=4):
    """Original KV cache memory in bytes (K + V, float32)."""
    return batch * heads * seq_len * head_dim * bytes_per_elem * 2

def kv_memory_compressed(batch, heads, seq_len, head_dim, bit_width):
    """Compressed KV cache memory in bytes.
    
    Each coordinate uses bit_width bits instead of 32.
    Plus we store one float32 norm per vector for both K and V.
    """
    n_vectors = batch * heads * seq_len
    # Bits for quantized data (K + V)
    data_bits = n_vectors * head_dim * bit_width * 2
    # Bits for norms (one float32 per vector, K + V)
    norm_bits = n_vectors * 32 * 2
    return (data_bits + norm_bits) // 8

# Example: LLaMA-70B-like config
batch_ex = 1
heads_ex = 64
head_dim_ex = 128

print(f"Config: batch={batch_ex}, heads={heads_ex}, head_dim={head_dim_ex}\n")
print(f"{'Seq Len':<12} {'Original':<14} {'3-bit Comp.':<14} {'Ratio':<10} {'Savings'}")
print("-" * 65)

seq_lens = [1_000, 10_000, 50_000, 100_000, 200_000]
orig_mb_list = []
comp_mb_list = []

for sl in seq_lens:
    orig = kv_memory_original(batch_ex, heads_ex, sl, head_dim_ex)
    comp = kv_memory_compressed(batch_ex, heads_ex, sl, head_dim_ex, bit_width=3)
    ratio = orig / comp
    orig_mb = orig / (1024 ** 2)
    comp_mb = comp / (1024 ** 2)
    orig_mb_list.append(orig_mb)
    comp_mb_list.append(comp_mb)
    print(f"  {sl:>9,}  {orig_mb:>10.1f} MB  {comp_mb:>10.1f} MB  {ratio:>6.1f}x    {(1 - comp/orig)*100:.0f}%")

In [ ]:
# Plot: memory usage vs sequence length
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[s / 1000 for s in seq_lens],
    y=orig_mb_list,
    mode="lines+markers",
    name="Original (float32)",
    line=dict(color="#e74c3c", width=3),
    marker=dict(size=8),
))

for bw, color, dash in [(2, "#2ecc71", "dot"), (3, "#3498db", "solid"), (4, "#9b59b6", "dash")]:
    comp_vals = [
        kv_memory_compressed(batch_ex, heads_ex, sl, head_dim_ex, bit_width=bw) / (1024**2)
        for sl in seq_lens
    ]
    fig.add_trace(go.Scatter(
        x=[s / 1000 for s in seq_lens],
        y=comp_vals,
        mode="lines+markers",
        name=f"Compressed ({bw}-bit)",
        line=dict(color=color, width=2, dash=dash),
        marker=dict(size=6),
    ))

# Add a "typical GPU budget" line
fig.add_hline(
    y=8 * 1024, line_dash="dash", line_color="gray",
    annotation_text="8 GB (typical inference budget)",
    annotation_position="top left",
)

fig.update_layout(
    title="KV Cache Memory vs. Sequence Length",
    xaxis_title="Sequence Length (thousands of tokens)",
    yaxis_title="Memory (MB)",
    template="plotly_white",
    height=500, width=800,
    legend=dict(x=0.02, y=0.98),
)
fig.show()

The plot makes the story clear: at 3 bits, you can serve roughly 8-9x longer sequences
in the same memory budget. For a 100K context window, that is the difference between
needing a single GPU and needing a cluster.

---
### 6. Reference implementation: TurboQuantKVCache

The reference code packages everything we just built into a clean class.
Here is the key structure:

```python
class TurboQuantKVCache:
    def __init__(self, head_dim, bit_width=3, seed=0):
        self.key_quantizer = TurboQuant(head_dim, bit_width, unbiased=True, seed=seed)
        self.value_quantizer = TurboQuant(head_dim, bit_width, unbiased=False, seed=seed + 100)

    def compress(self, keys, values):
        B, H, S, D = keys.shape
        k_flat = keys.reshape(-1, D)
        v_flat = values.reshape(-1, D)
        k_compressed = self.key_quantizer.quantize(k_flat)
        v_compressed = self.value_quantizer.quantize(v_flat)
        return CompressedKV(keys=k_compressed, values=v_compressed, ...)

    def attention(self, query, compressed, scale=None):
        keys = self.decompress_keys(compressed)
        values = self.decompress_values(compressed)
        attn_weights = torch.matmul(query, keys.transpose(-2, -1)) * scale
        attn_weights = F.softmax(attn_weights, dim=-1)
        return torch.matmul(attn_weights, values)
```

**Why keys get `unbiased=True` and values get `unbiased=False`:**

| Component | What it feeds into | What matters | Setting |
|-----------|-------------------|--------------|----------|
| **Keys** | `Q @ K^T` (inner product) | Accurate dot products for correct attention routing | `unbiased=True` |
| **Values** | Weighted sum (`attn_weights @ V`) | Low reconstruction error for output quality | `unbiased=False` |

Keys participate in a **dot product** -- the attention logit. A biased estimate would
systematically shift which tokens get attended to. The QJL correction (unbiased mode)
ensures the expected value of the estimated logit equals the true logit.

Values participate in a **weighted average**. What matters here is that the reconstructed
value vectors are close to the originals (low MSE). The unbiased correction would add
noise without helping, since values are not used in a dot-product comparison.

In [ ]:
# Verify with the reference implementation
from turboquant.kv_cache import TurboQuantKVCache

kv_cache = TurboQuantKVCache(head_dim=head_dim, bit_width=3, seed=0)
compressed = kv_cache.compress(keys, values)
attn_ref = kv_cache.attention(query, compressed)

ref_mse = (attn_output_orig - attn_ref).pow(2).mean().item()
ref_cos = F.cosine_similarity(
    attn_output_orig.reshape(-1, D), attn_ref.reshape(-1, D), dim=-1
).mean().item()

print(f"Reference implementation results (3-bit):")
print(f"  Output MSE:             {ref_mse:.6f}")
print(f"  Output cosine sim:      {ref_cos:.6f}")

orig_mb, comp_mb, ratio = kv_cache.memory_savings(batch, heads, seq_len)
print(f"\nMemory: {orig_mb:.2f} MB -> {comp_mb:.2f} MB ({ratio:.1f}x compression)")

---
## Part 2: Vector Search

### 7. TurboQuant as an ANN index

Approximate nearest neighbor (ANN) search is the backbone of retrieval systems:
semantic search, recommendation engines, RAG pipelines. The standard approach is
**product quantization (PQ)**:

1. Split each vector into subspaces
2. Run **k-means** in each subspace to learn a codebook
3. Encode each vector as a sequence of codebook indices
4. Search by computing approximate distances using lookup tables

PQ works well, but step 2 is the bottleneck: **k-means training is slow** and
requires a representative dataset. If your data distribution shifts, you need
to retrain.

TurboQuant eliminates this entirely:

- **Indexing time: instant.** Just quantize the vectors. No k-means, no training.
- **Works on any data.** Data-oblivious -- no codebook to fit.
- **Preserves inner products.** With `unbiased=True`, the estimated inner product
  is unbiased, which means rankings are preserved on average.

Let's measure how well it works.

In [ ]:
section_header(
    "Vector Search Experiment",
    "10,000 vectors, measure recall@10 at various bit widths"
)

# Create a database of random "embedding" vectors
torch.manual_seed(42)
n_db = 10_000
n_queries = 100
dim = 256

# Simulate normalized embeddings (common in practice)
db_vectors = torch.randn(n_db, dim)
db_vectors = F.normalize(db_vectors, dim=-1)

query_vectors = torch.randn(n_queries, dim)
query_vectors = F.normalize(query_vectors, dim=-1)

print(f"Database:  {n_db:,} vectors, dim={dim}")
print(f"Queries:   {n_queries} vectors")
print(f"Database size (float32): {n_db * dim * 4 / (1024**2):.1f} MB")

In [ ]:
# Ground truth: exact inner product search
k = 10
exact_scores = query_vectors @ db_vectors.t()  # (n_queries, n_db)
_, exact_topk = torch.topk(exact_scores, k, dim=-1)  # (n_queries, k)

print(f"Ground truth: top-{k} neighbors computed via exact inner product.")
print(f"Score range: [{exact_scores.min():.3f}, {exact_scores.max():.3f}]")

In [ ]:
# Search with TurboQuant at different bit widths

def recall_at_k(pred_indices, true_indices):
    """Compute recall@k: fraction of true top-k found in predicted top-k."""
    recalls = []
    for p, t in zip(pred_indices, true_indices):
        p_set = set(p.tolist())
        t_set = set(t.tolist())
        recalls.append(len(p_set & t_set) / len(t_set))
    return np.mean(recalls)

search_results = {}

for bw in [2, 3, 4]:
    quantizer = TurboQuant(dim, bit_width=bw, unbiased=True, seed=0)

    # Index (quantize database)
    t0 = time.perf_counter()
    db_compressed = quantizer.quantize(db_vectors)
    index_time = time.perf_counter() - t0

    # Search (compute inner products with compressed database)
    t0 = time.perf_counter()
    approx_scores = quantizer.compute_inner_product(query_vectors, db_compressed)
    _, approx_topk = torch.topk(approx_scores, k, dim=-1)
    search_time = time.perf_counter() - t0

    rec = recall_at_k(approx_topk, exact_topk)

    # Memory
    compressed_bytes = quantizer.memory_bytes(n_db)
    original_bytes = n_db * dim * 4
    ratio = original_bytes / compressed_bytes

    search_results[bw] = {
        "recall": rec,
        "index_time": index_time,
        "search_time": search_time,
        "ratio": ratio,
        "compressed_mb": compressed_bytes / (1024**2),
    }

    print(f"{bw}-bit: recall@{k}={rec:.3f}, index={index_time*1000:.1f}ms, "
          f"search={search_time*1000:.1f}ms, compression={ratio:.1f}x")

In [ ]:
# Compare indexing time: TurboQuant vs simulated PQ training time
# PQ typically needs k-means with multiple iterations over the data.
# We simulate the cost of k-means on subspaces.

section_header(
    "Indexing Time Comparison",
    "TurboQuant (instant) vs. Product Quantization (k-means training)"
)

# Simulate PQ training: split into subspaces, run k-means iterations
n_subspaces = 32
sub_dim = dim // n_subspaces
n_centroids = 256  # 8-bit PQ
n_iters = 20  # typical k-means iterations

t0 = time.perf_counter()
for s in range(n_subspaces):
    sub_data = db_vectors[:, s * sub_dim : (s + 1) * sub_dim]
    # Simulate k-means: random init + assignment + update, n_iters times
    centroids = sub_data[torch.randperm(n_db)[:n_centroids]].clone()
    for _ in range(n_iters):
        # Assignment step: compute distances to all centroids
        dists = torch.cdist(sub_data, centroids)  # (n_db, n_centroids)
        assignments = dists.argmin(dim=-1)  # (n_db,)
        # Update step: recompute centroids
        for c in range(n_centroids):
            mask = assignments == c
            if mask.any():
                centroids[c] = sub_data[mask].mean(dim=0)
pq_train_time = time.perf_counter() - t0

# TurboQuant indexing time at 3 bits
tq_index_time = search_results[3]["index_time"]

print(f"Product Quantization training: {pq_train_time*1000:.0f} ms")
print(f"TurboQuant indexing (3-bit):   {tq_index_time*1000:.1f} ms")
print(f"\nSpeedup: {pq_train_time / tq_index_time:.0f}x faster")
print("\nAnd PQ training gets WORSE with more data.")
print("TurboQuant indexing scales linearly -- just quantize each vector.")

---
### 8. Reference implementation: TurboQuantIndex

The reference code wraps TurboQuant into a clean search index API:

```python
class TurboQuantIndex:
    def __init__(self, dim, bit_width=3, metric="ip", seed=0):
        self.quantizer = TurboQuant(dim, bit_width, unbiased=True, seed=seed)

    def add(self, vectors):
        # Instant -- just quantize, no training
        self._db = self.quantizer.quantize(vectors)

    def search(self, query, k=10):
        scores = self.quantizer.compute_inner_product(query, self._db)
        return torch.topk(scores, k, dim=-1)
```

Three lines of real logic. The simplicity is the point: TurboQuant's data-oblivious
design means the index is just "quantize and search." No codebook, no training loop,
no hyperparameter tuning.

For inner product search, `unbiased=True` is essential. The QJL correction ensures
that E[estimated_ip] = true_ip, which keeps the ranking correct on average.

In [ ]:
# Verify with the reference implementation
from turboquant.vector_search import TurboQuantIndex

index = TurboQuantIndex(dim=dim, bit_width=3, metric="ip", seed=0)
index.add(db_vectors)

ref_scores, ref_indices = index.search(query_vectors, k=10)
ref_recall = recall_at_k(ref_indices, exact_topk)

print(f"Reference TurboQuantIndex (3-bit):")
print(f"  recall@10 = {ref_recall:.3f}")
print(f"  Memory:     {index.memory_bytes() / (1024**2):.2f} MB")
print(f"  (vs original {n_db * dim * 4 / (1024**2):.2f} MB)")

---
### 9. Performance summary

Let's bring everything together in a single table.

In [ ]:
section_header(
    "Performance Summary",
    "Both applications at a glance"
)

print("=" * 80)
print("KV CACHE COMPRESSION (batch=1, heads=8, seq=512, head_dim=128)")
print("=" * 80)
print(f"{'Bits':<8} {'Compression':<14} {'Attn Weight CosSim':<22} {'Output MSE':<16} {'Output CosSim'}")
print("-" * 80)
for bw, r in results_kv.items():
    ratio = 32.0 / bw
    print(f"  {bw:<6} {ratio:>6.1f}x        {r['weight_cos']:<22.4f} {r['output_mse']:<16.6f} {r['output_cos']:.4f}")

print()
print("=" * 80)
print(f"VECTOR SEARCH (n={n_db:,}, dim={dim}, queries={n_queries})")
print("=" * 80)
print(f"{'Bits':<8} {'Compression':<14} {'Recall@10':<14} {'Index Time':<16} {'Search Time'}")
print("-" * 80)
for bw, r in search_results.items():
    print(f"  {bw:<6} {r['ratio']:>6.1f}x        {r['recall']:<14.3f} {r['index_time']*1000:<13.1f} ms  {r['search_time']*1000:.1f} ms")

print()
print("Key insight: 3 bits is the sweet spot for both applications.")
print("  - KV cache: ~10x compression with negligible quality loss")
print("  - Search:   ~10x compression with high recall, instant indexing")

In [ ]:
# Final visualization: recall vs compression for vector search
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Recall@10 vs. Bit Width", "Indexing Time Comparison"),
    horizontal_spacing=0.15,
)

bit_widths = list(search_results.keys())
recalls = [search_results[bw]["recall"] for bw in bit_widths]
index_times = [search_results[bw]["index_time"] * 1000 for bw in bit_widths]

fig.add_trace(
    go.Scatter(
        x=bit_widths, y=recalls, mode="lines+markers+text",
        name="TurboQuant",
        line=dict(color="#3498db", width=3),
        marker=dict(size=12),
        text=[f"{r:.2f}" for r in recalls],
        textposition="top center",
    ),
    row=1, col=1,
)
fig.update_yaxes(range=[0, 1.05], title_text="Recall@10", row=1, col=1)
fig.update_xaxes(title_text="Bits per coordinate", tickvals=[2, 3, 4], row=1, col=1)

# Indexing time comparison
fig.add_trace(
    go.Bar(
        x=["PQ Training"],
        y=[pq_train_time * 1000],
        name="Product Quantization",
        marker_color="#e74c3c",
        text=[f"{pq_train_time*1000:.0f} ms"],
        textposition="outside",
        width=0.4,
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Bar(
        x=["TQ Index (3-bit)"],
        y=[tq_index_time * 1000],
        name="TurboQuant",
        marker_color="#3498db",
        text=[f"{tq_index_time*1000:.1f} ms"],
        textposition="outside",
        width=0.4,
    ),
    row=1, col=2,
)
fig.update_yaxes(title_text="Time (ms)", row=1, col=2)

fig.update_layout(
    title="Vector Search: TurboQuant Performance",
    template="plotly_white",
    height=450, width=900,
    showlegend=False,
)
fig.show()

---
## Summary

This module demonstrated two practical applications of TurboQuant:

In [ ]:
section_header(
    "Summary: Applications of TurboQuant",
    "Module 11 -- KV Cache & Vector Search"
)

print("KV Cache Compression:")
print("  - Keys use unbiased=True  (accurate attention logits via inner products)")
print("  - Values use unbiased=False (low MSE for output reconstruction)")
print("  - 3-bit compression gives ~10x memory savings with minimal quality loss")
print("  - No training needed: works with any model, any sequence length")
print("  - Enables longer context windows within fixed GPU memory budgets")
print()
print("Vector Search:")
print("  - Zero indexing time: just quantize the vectors (no k-means training)")
print("  - High recall@10 at 3-4 bits with ~10x memory compression")
print("  - Data-oblivious: no codebook to retrain when data distribution shifts")
print("  - Inner product preservation (unbiased=True) keeps rankings accurate")
print()
print("The connecting thread: both applications rely on preserving inner products,")
print("which is exactly what TurboQuant was designed to do.")

---

*You have now seen TurboQuant in action on two real-world problems. The same algorithm
that started as random rotations and scalar quantization turns out to be a practical
tool for some of the most memory-intensive operations in modern ML.*